# DETR/YOLOS Nesne Tespiti Notebook'u / DETR/YOLOS Object Detection Notebook

Bu notebook, HuggingFace transformers kütüphanesinden DETR ve YOLOS modellerini kullanarak nesne tespitinin nasıl yapılacağını gösterir.
This notebook demonstrates how to perform object detection using DETR and YOLOS models from HuggingFace transformers library.

## DETR Nedir? / What is DETR?

DETR (DEtection TRansformer), Facebook AI Research tarafından geliştirilen yenilikçi bir nesne tespit modelidir.
DETR (DEtection TRansformer) is an innovative object detection model developed by Facebook AI Research.

### Özellikleri / Features:
- **End-to-end**: Geleneksel pipeline'dan farklı olarak doğrudan sonuç üretir
- **Transformer tabanlı**: Attention mekanizması kullanır
- **Zero-shot**: COCO dışındaki sınıflara genişletilebilir
- **Set prediction**: Tüm nesneleri tek seferde tahmin eder

In [ ]:
# Gerekli kütüphaneleri içe aktar / Import required libraries
import os
import sys
from pathlib import Path

# Proje kök dizinini path'e ekle / Add project root to path
project_root = Path('../').resolve()
sys.path.insert(0, str(project_root))

import cv2
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import torch

print("Kütüphaneler başarıyla içe aktarıldı!")
print(f"PyTorch sürümü: {torch.__version__}")
print(f"CUDA mevcut: {torch.cuda.is_available()}")

## DETR Model Yükleme / Loading DETR Model

In [ ]:
from transformers import AutoImageProcessor, AutoModelForObjectDetection

MODEL_ID = "facebook/detr-resnet-50"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print(f"DETR modeli yükleniyor: {MODEL_ID}")
processor = AutoImageProcessor.from_pretrained(MODEL_ID)
model = AutoModelForObjectDetection.from_pretrained(MODEL_ID)
model.to(DEVICE)
model.eval()
print("DETR modeli başarıyla yüklendi!")

## COCO Sınıf Etiketleri / COCO Class Labels

In [ ]:
COCO_CLASSES = [
    'person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck',traffic light',
    'fire hydrant', 'stop 'boat', ' sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow',
    'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee',
    'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle',
    'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange',
    'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'couch', 'potted plant', 'bed',
    'dining table', 'toilet', 'tv', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven',
    'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush'
]
print(f"Toplam sınıf sayısı: {len(COCO_CLASSES)}")

## Test Görüntüsü Oluşturma / Creating Test Image

In [ ]:
def create_industrial_test_image():
    """Endüstriyel test görüntüsü oluştur"""
    img = np.zeros((640, 640, 3), dtype=np.uint8)
    img[:] = (80, 80, 85)  # Arka plan
    # İnsan (person)
    cv2.ellipse(img, (200, 300), (40, 80), 0, 0, 180, (200, 150, 100), -1)
    cv2.circle(img, (200, 180), 30, (220, 170, 120), -1)
    # Kutu (box)
    cv2.rectangle(img, (350, 400), (500, 550), (120, 80, 60), -1)
    # Araç (truck)
    cv2.rectangle(img, (50, 380), (250, 520), (150, 100, 80), -1)
    # Şişe (bottle)
    cv2.rectangle(img, (520, 350), (560, 450), (100, 150, 200), -1)
    return img

test_image = create_industrial_test_image()
image_pil = Image.fromarray(test_image)
plt.figure(figsize=(10, 10))
plt.imshow(test_image)
plt.title("Endüstriyel Test Görüntüsü")
plt.axis('off')
plt.show()

## DETR ile Nesne Tespiti / Object Detection with DETR

In [ ]:
def detect_objects_detr(image, model, processor, threshold=0.5):
    """DETR ile nesne tespiti yapar."""
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=threshold
    )[0]
    return results

print("DETR ile nesne tespiti yapılıyor...")
results = detect_objects_detr(image_pil, model, processor, threshold=0.5)
print(f"Tespit edilen nesne sayısı: {len(results['scores'])}")
for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
    box = box.tolist()
    print(f"  - {COCO_CLASSES[label.item()]:15s} | Güven: {score.item():.3f}")

## Sonuçları Görselleştirme / Visualizing Results

In [ ]:
def visualize_detr_results(image, results, class_names, threshold=0.5):
    """DETR sonuçlarını görselleştirir."""
    img = np.array(image).copy()
    colors = plt.cm.hsv(np.linspace(0, 1, 80)).tolist()
    for score, label, box in zip(results['scores'], results['labels'], results['boxes']):
        if score.item() < threshold:
            continue
        box = box.tolist()
        label_idx = label.item()
        class_name = class_names[label_idx]
        x1, y1, x2, y2 = int(box[0]), int(box[1]), int(box[2]), int(box[3])
        color = tuple(int(c * 255) for c in colors[label_idx][:3])
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        label_text = f"{class_name}: {score.item():.2f}"
        cv2.putText(img, label_text, (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 2)
    return img

result_img = visualize_detr_results(image_pil, results, COCO_CLASSES)
plt.figure(figsize=(12, 12))
plt.imshow(result_img)
plt.title("DETR ile Tespit Edilen Nesneler")
plt.axis('off')
plt.show()

## YOLOS Model Yükleme / Loading YOLOS Model

In [ ]:
YOLOS_MODEL_ID = "hustvl/yolos-small"
print(f"YOLOS modeli yükleniyor: {YOLOS_MODEL_ID}")
yolos_processor = AutoImageProcessor.from_pretrained(YOLOS_MODEL_ID)
yolos_model = AutoModelForObjectDetection.from_pretrained(YOLOS_MODEL_ID)
yolos_model.to(DEVICE)
yolos_model.eval()
print("YOLOS modeli başarıyla yüklendi!")

In [ ]:
def detect_objects_yolos(image, model, processor, threshold=0.5):
    """YOLOS ile nesne tespiti yapar."""
    inputs = processor(images=image, return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        outputs = model(**inputs)
    target_sizes = torch.tensor([image.size[::-1]])
    results = processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=threshold
    )[0]
    return results

print("YOLOS ile nesne tespiti yapılıyor...")
yolos_results = detect_objects_yolos(image_pil, yolos_model, yolos_processor, threshold=0.5)
print(f"Tespit edilen nesne sayısı: {len(yolos_results['scores'])}")
for score, label, box in zip(yolos_results['scores'], yolos_results['labels'], yolos_results['boxes']):
    box = box.tolist()
    print(f"  - {COCO_CLASSES[label.item()]:15s} | Güven: {score.item():.3f}")

In [ ]:
yolos_result_img = visualize_detr_results(image_pil, yolos_results, COCO_CLASSES)
plt.figure(figsize=(12, 12))
plt.imshow(yolos_result_img)
plt.title("YOLOS ile Tespit Edilen Nesneler")
plt.axis('off')
plt.show()

## Karşılaştırma / Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 9))
axes[0].imshow(result_img)
axes[0].set_title(f"DETR: {len(results['scores'])} nesne")
axes[0].axis('off')
axes[1].imshow(yolos_result_img)
axes[1].set_title(f"YOLOS: {len(yolos_results['scores'])} nesne")
axes[1].axis('off')
plt.tight_layout()
plt.show()

## Performans Karşılaştırması / Performance Comparison

In [ ]:
import time

def measure_inference(model, processor, image, model_name):
    """Inference süresini ölçer."""
    # Warmup
    for _ in range(3):
        _ = processor(images=image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            _ = model(**processor(images=image, return_tensors="pt").to(DEVICE))
    # Gerçek ölçüm
    start = time.time()
    iterations = 20
    for _ in range(iterations):
        inputs = processor(images=image, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            _ = model(**inputs)
    end = time.time()
    return (end - start) / iterations

detr_time = measure_inference(model, processor, image_pil, "DETR")
yolos_time = measure_inference(yolos_model, yolos_processor, image_pil, "YOLOS")

print(f"DETR ortalama inference süresi: {detr_time*1000:.2f} ms")
print(f"YOLOS ortalama inference süresi: {yolos_time*1000:.2f} ms")
print(f"Hız oranı / Speed ratio: {detr_time/yolos_time:.2f}x")

## Özet / Summary

Bu notebook'ta:
- DETR ve YOLOS modellerini yüklemeyi öğrendik
- Nesne tespiti yapmayı gördük
- Performans karşılaştırması yaptık

### Sonraki Adımlar:
- SAM segmentasyon notebook'una geçin
- Model karşılaştırması yapın
- Endüstriyel uygulamaları keşfedin